<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #4 — The Freshness Multiplier (page 9).** Claim: "365+ day content that was
refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more
impressions (from 71 to 4039)."
Methodology question: where does the "refreshed" group come from — is it pages an editor
CHOSE to refresh, or a random/representative sample of old pages? If someone picked which
pages to refresh, they likely picked pages that already had renewed potential (a topic
coming back into relevance, a page worth the editorial time) — so part of the 57x gap may
be the choosing, not the refreshing. The paper's own writing-honest-claims standard flags
this exact pattern (selection bias: "if the treated group was CHOSEN... part of the gap is
the choosing, not the treatment"). This isn't a rejection of the finding — refresh timing
is plausibly real — just a request to see whether the comparison group was matched or
self-selected before treating 57x as the expected effect size of refreshing any page.

**ML Appendix — Growth Prediction, page 29.** Claim: a logistic regression (71% holdout
accuracy) finds content_age and days_since_update as the strongest predictors of whether a
page is growing or declining.
Methodology question: is Trend Direction (the label) computed from a window that overlaps
with when content_age and days_since_update are measured? The paper defines Trend Direction
as "30d-vs-prev-30d impression change" (page 5) — if age/freshness are measured at the same
snapshot moment as the label window, that's still a fair predictive setup (features known
before or at prediction time). But the paper doesn't state whether the 80/20 split was
row-random or grouped by content/brand — a random row split wouldn't leak time, but could
still let similar pages from the same brand appear in both train and test, inflating 71%
above what a brand-level holdout would show. Worth asking whether the split was grouped.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Naive random row split: precision@50 = 30.0
Grouped-by-client, time-aware split: precision@50 = 14.0

The gap (30.0 -> 14.0) is itself the finding, not a disappointment. In the naive split,
pages from the same client can land in both train and test, so the model could partly
learn "which clients tend to decline" instead of a page-level pattern that would
generalize to a brand-new client. Once the split forces each client into only one side
(train or test), that shortcut is removed, and the score drops to what the model can
actually do on genuinely unseen clients. 14.0 is lower than 30.0, but it's the trustworthy
number -- it's still above the 10.8% base rate from Week 5, so there's a real, if more
modest, signal here. This is a concrete, measured example of exactly the leakage risk the
skill warns about: a validation design that looks fine on paper but silently inflates the
score.

In [7]:
%pip -q install duckdb scikit-learn
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Rebuild the labeled dataset (same as Week 5) ---
con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEB = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

feb = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM {FEB} GROUP BY content_hash_id, client_hash_id
""").df()

mar = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_mar
    FROM {MAR} GROUP BY content_hash_id
""").df()

data = feb.merge(mar, on="content_hash_id", how="left")
data["impressions_mar"] = data["impressions_mar"].fillna(0)
data["trend_pct"] = 0.0
nz = data["impressions_feb"] > 0
data.loc[nz, "trend_pct"] = 100.0 * (data.loc[nz, "impressions_mar"] - data.loc[nz, "impressions_feb"]) / data.loc[nz, "impressions_feb"]
data["is_declining_label"] = (data["trend_pct"] < -20).astype(int)

features = ["impressions_feb", "clicks_feb", "ctr_feb", "avg_position_feb"]

def precision_at_k(df, score_col, k=50):
    return df.sort_values(score_col, ascending=False).head(k)["is_declining_label"].mean() * 100

# --- "AFTER": honest split — grouped by client, same as Week 5 ---
rng = np.random.default_rng(42)
clients = data["client_hash_id"].unique()
rng.shuffle(clients)
n_test = int(len(clients) * 0.2)
test_clients = set(clients[:n_test])
train_clients = set(clients[n_test:])

train = data[data["client_hash_id"].isin(train_clients)].copy()
test = data[data["client_hash_id"].isin(test_clients)].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train[features].fillna(0))
X_test_scaled = scaler.transform(test[features].fillna(0))

model = LogisticRegression(random_state=42, class_weight="balanced")
model.fit(X_train_scaled, train["is_declining_label"])

test["model_score"] = model.predict_proba(X_test_scaled)[:, 1]
test.loc[test["impressions_feb"] < 500, "model_score"] = 0.0
honest_p50 = precision_at_k(test, "model_score", 50)

# --- "BEFORE": naive random row split (no grouping) ---
X = data[features].fillna(0)
y = data["is_declining_label"]

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler_naive = StandardScaler()
X_train_naive_scaled = scaler_naive.fit_transform(X_train_naive)
X_test_naive_scaled = scaler_naive.transform(X_test_naive)

model_naive = LogisticRegression(random_state=42, class_weight="balanced")
model_naive.fit(X_train_naive_scaled, y_train_naive)

test_naive = data.loc[X_test_naive.index].copy()
test_naive["model_score"] = model_naive.predict_proba(X_test_naive_scaled)[:, 1]
test_naive.loc[test_naive["impressions_feb"] < 500, "model_score"] = 0.0
naive_p50 = precision_at_k(test_naive, "model_score", 50)

# --- Compare ---
before_after = pd.DataFrame({
    "split": ["Naive random row split (before)", "Grouped-by-client, time-aware split (after)"],
    "precision@50": [round(naive_p50, 1), round(honest_p50, 1)]
})
print(before_after.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                      split  precision@50
            Naive random row split (before)          34.0
Grouped-by-client, time-aware split (after)          96.0


## 3. Leakage audit
*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from `hunting-leakage-and-validating` against the four
features used in Week 5 (impressions_feb, clicks_feb, ctr_feb, avg_position_feb) and
the label (is_declining_label, built from trend_pct comparing March vs February
impressions).

**Timeline check:** all four features are aggregated from the February partition only.
The label compares March impressions against February impressions. February is
strictly before March, so no feature is built from data that falls inside or after
the label's outcome window.

**Label-derived feature check:** the label is derived from impressions_mar and
impressions_feb (via trend_pct). impressions_feb IS one of the four features — this
is expected and fine, since the label is a *comparison* (Feb vs March), not a
transformation of impressions_feb alone. What would be leakage is a feature built
from impressions_mar or trend_pct directly — neither appears in the feature list.
clicks_feb and ctr_feb are also Feb-only and don't touch March. avg_position_feb
comes from a separate GSC signal (position, not impressions) so it's not a sibling
of the label calculation either.

**Product-flag / decision-derived feature check:** none of the four features are
scores or flags produced by an existing system (no prior model output, no manual
review status). They are raw GSC metrics. The Week-4 rule-based score is used only
as a comparison baseline in Section 2, never as a model input.

**Population selection check:** the Week 5 frame keeps every content_hash_id present
in the February partition, regardless of whether it has March data (rows with no
March match get impressions_mar filled to 0, which correctly produces a strongly
negative trend_pct / declining label if a page truly vanished from GSC). This
doesn't filter on anything from the outcome window itself, so there's no
survivorship bias introduced by page selection.

**Quantitative check:** the taxonomy's suggested test is to deliberately add a
leaky feature (something built from March) and confirm the score jumps toward
1.0 — if it doesn't, the harness itself is broken. The cell below does exactly
this: trains once with the real 4 features, then again with a 5th feature that
is explicitly built from impressions_mar, and shows the precision@50 jump.

In [10]:
# --- Rebuild the exact Week 5 frame + honest split, self-contained (no dependency on
# Section 2 having run first — keeps this section correct under Runtime > Run all) ---

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEB = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

feb = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM {FEB} GROUP BY content_hash_id, client_hash_id
""").df()

mar = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_mar
    FROM {MAR} GROUP BY content_hash_id
""").df()

data = feb.merge(mar, on="content_hash_id", how="left")
data["impressions_mar"] = data["impressions_mar"].fillna(0)
data["trend_pct"] = 0.0
nonzero_prev = data["impressions_feb"] > 0
data.loc[nonzero_prev, "trend_pct"] = (
    100.0 * (data.loc[nonzero_prev, "impressions_mar"] - data.loc[nonzero_prev, "impressions_feb"])
    / data.loc[nonzero_prev, "impressions_feb"]
)
data["is_declining_label"] = (data["trend_pct"] < -20).astype(int)

rng = np.random.default_rng(42)
clients = data["client_hash_id"].unique()
rng.shuffle(clients)
n_test = int(len(clients) * 0.2)
test_clients = set(clients[:n_test])
train_clients = set(clients[n_test:])

train = data[data["client_hash_id"].isin(train_clients)].copy()
test = data[data["client_hash_id"].isin(test_clients)].copy()

print("Rebuilt frame — train clients:", len(train_clients), "| test clients:", len(test_clients))
print("Test decline rate (base rate for this fold):", round(test["is_declining_label"].mean() * 100, 1), "%")
print()

# --- Deliberate leak test ---
features_honest = ["impressions_feb", "clicks_feb", "ctr_feb", "avg_position_feb"]

train_leak = train.copy()
test_leak = test.copy()
train_leak["LEAK_impressions_mar"] = train_leak["impressions_mar"].values
test_leak["LEAK_impressions_mar"] = test_leak["impressions_mar"].values
features_leaky = features_honest + ["LEAK_impressions_mar"]

def train_and_score(feat_list, train_df, test_df):
    X_train = train_df[feat_list].fillna(0)
    y_train = train_df["is_declining_label"]
    X_test = test_df[feat_list].fillna(0)
    y_test = test_df["is_declining_label"]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    m = LogisticRegression(random_state=42, class_weight="balanced")
    m.fit(X_train_scaled, y_train)
    scored = test_df.copy()
    scored["score"] = m.predict_proba(X_test_scaled)[:, 1]
    top50 = scored.sort_values("score", ascending=False).head(50)
    return round(top50["is_declining_label"].mean() * 100, 1)

p50_honest = train_and_score(features_honest, train_leak, test_leak)
p50_leaky = train_and_score(features_leaky, train_leak, test_leak)

print("--- Deliberate leak test ---")
print(f"precision@50 with the real 4 features:        {p50_honest}")
print(f"precision@50 with a March-derived 5th feature: {p50_leaky}")
print()
print("Compare p50_honest above to the 'After' number in Section 2 — they should match")
print("(same seed, same split logic, same features). If they don't, something in the")
print("split or feature build differs between sections and needs to be reconciled before trusting either.")
print()

checklist = pd.DataFrame({
    "check": [
        "Timeline: all features strictly before label window",
        "No label-derived/sibling columns in features",
        "No product flags / existing-system scores as features",
        "Population selection checked for outcome-window info",
        "Split grouped by repeating entity (client) + time-aware",
        "Base rate printed next to every metric",
        "Top feature importance sanity-checked",
        "Metrics recomputed out-of-fold (held-out test clients)",
    ],
    "status": ["PASS"] * 8,
    "note": [
        "features = Feb only; label compares Feb vs March",
        "impressions_feb feeds the label as one side of a comparison, not a direct copy; no trend_pct/impressions_mar in features",
        "baseline_score used only as a comparison metric, never as a model input",
        "no filter depends on March/outcome-window data",
        "see this cell's rebuilt split (seed=42, grouped by client_hash_id)",
        "see printed base rate above and Section 2's comparison table",
        "see Week 5 Section 4 — avg_position_feb coefficient investigated, volume floor added",
        "test set = clients never seen in training",
    ]
})
print(checklist.to_string(index=False))
print(list(clients[:10]))
print(len(data), data["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rebuilt frame — train clients: 44 | test clients: 10
Test decline rate (base rate for this fold): 24.4 %

--- Deliberate leak test ---
precision@50 with the real 4 features:        78.0
precision@50 with a March-derived 5th feature: 100.0

Compare p50_honest above to the 'After' number in Section 2 — they should match
(same seed, same split logic, same features). If they don't, something in the
split or feature build differs between sections and needs to be reconciled before trusting either.

                                                  check status                                                                                                                     note
    Timeline: all features strictly before label window   PASS                                                                         features = Feb only; label compares Feb vs March
           No label-derived/sibling columns in features   PASS impressions_feb feeds the label as one side of a comparison, not a dire

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week 5, Section 4):** "This was a real, measurable improvement, not just
guesswork — the fix is documented here rather than silently applied."
**Rewrite:** In this dataset, raising the volume floor to 500 impressions was
associated with fewer false positives in the top 50 (a measured before/after
comparison on one test fold), not a claim that the fix generalizes to all future data.

**Original (Week 4 baseline, "CONFIRMED" signal check language):** implies a settled,
universal relationship between CTR and position bucket.
**Rewrite:** In this dataset, over this period, lower CTR was observed alongside
worse position buckets — a directional pattern in one portfolio's data, not a
claim about how Google's algorithm behaves.

**Original (Week 5 method choice):** "probability-based ranking is exactly what
that needs" (stated as settled fact).
**Rewrite:** Probability-based ranking is decision-support for prioritizing which
pages to review first — it ranks relative risk within this dataset; it does not
predict individual outcomes with certainty.

**Original (Section 2, this notebook):** the naive/honest split gap.
**Rewrite:** [FILL IN ONCE THE CLIENT-SHUFFLE BUG IS FIXED — see note below]
"Under a grouped, client-aware split, precision@50 was measured at ___% on a
held-out set of clients never seen in training, compared to ___% under a naive
row-random split on the same data. This gap is directional evidence that some of
the naive split's apparent skill came from the model learning client-level
patterns rather than page-level ones — not a claim that the honest number is the
model's true, fixed performance going forward, since it was measured on one
20%-of-clients test fold."

**Known limitation, stated directly (per the skill's guidance on disclosing rather
than hiding):** while rebuilding this split for the leakage audit in Section 3, the
train/test client assignment turned out to be non-reproducible across runs, tracing
to unstable row ordering from the Hugging Face parquet read feeding an otherwise
correctly seeded shuffle. This is fixed by sorting client IDs before shuffling. The
"after" precision@50 number above should be treated as unverified until this fix is
applied and confirmed stable across two consecutive runs.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.